https://dados.cvm.gov.br/dataset/cia_aberta-doc-dfp

💰 Lucro Líquido: dfp_cia_aberta_DRE_con_XXXX.csv (Conta 3.11)

🏢 Patrimônio Líquido: dfp_cia_aberta_BPP_con_XXXX.csv (Conta 2.03)

📈 Número de Ações: dfp_cia_aberta_composicao_capital_XXXX.csv

In [ ]:
import sys
import os
sys.path
os.listdir()
os.chdir(os.getcwd().replace("\\","/").replace("/notebooks",""))
sys.path.append("src")

In [ ]:
from src.envConfig import EnvConfig
EnvConfig()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, substring, trim, expr, when, lit

In [ ]:
# 1. Inicializa a sessão Spark
spark = SparkSession.builder \
    .appName("Cias Abertas: DFP") \
    .getOrCreate()

In [ ]:
CON = "asserts/dfp_cia_aberta_composicao_capital_2025.csv"
BPP_CON = "asserts/dfp_cia_aberta_BPP_con_2025.csv"
DRE = "asserts/dfp_cia_aberta_DRE_con_2025.csv"

In [ ]:
df_bpp = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ";") \
    .load(CON)
    
df_bpp_con = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ";") \
    .load(BPP_CON)
    
df_dre_con = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ";") \
    .load(DRE)

In [ ]:
df_bpp.show(truncate=False)

In [ ]:
df_bpp_con.show(truncate=False)

In [ ]:
df_dre_con.show(truncate=False)

In [ ]:
df_dre_con.describe()

In [ ]:
df_dre_con = (
    df_dre_con
    .filter(col("CD_CONTA") == "3.11")
)

df_bpp_con = (
    df_bpp_con
    .filter(col("CD_CONTA") == "2.03")
)

In [ ]:
lucro_acao = df_dre_con.join(df_bpp, on="CNPJ_CIA", how='inner')
valor_acao = df_bpp_con.join(df_bpp, on="CNPJ_CIA", how='inner')

In [ ]:
lucro_acao = (
    lucro_acao
    .withColumn("LPA - total", col("VL_CONTA")/col("QT_ACAO_TOTAL_CAP_INTEGR"))
    .withColumn(
            "LPA - pre", 
            when(col("QT_ACAO_PREF_CAP_INTEGR") == 0, lit(0))
            .otherwise(col("VL_CONTA")/col("QT_ACAO_PREF_CAP_INTEGR"))
        )
    .withColumn(
            "LPA - on",
            when(col("QT_ACAO_ORDIN_CAP_INTEGR") == 0, lit(0))
            .otherwise(col("VL_CONTA")/col("QT_ACAO_ORDIN_CAP_INTEGR"))
        )
)

valor_acao = (
    valor_acao
    .withColumn("VPA - total", col("VL_CONTA")/col("QT_ACAO_TOTAL_CAP_INTEGR"))
    .withColumn(
        "VPA - pre", 
        when(col("QT_ACAO_PREF_CAP_INTEGR") == 0, lit(0))
        .otherwise(col("VL_CONTA")/col("QT_ACAO_PREF_CAP_INTEGR"))
    )
    .withColumn(
            "VPA - on",
            when(col("QT_ACAO_ORDIN_CAP_INTEGR") == 0, lit(0))
            .otherwise(col("VL_CONTA")/col("QT_ACAO_ORDIN_CAP_INTEGR"))
        )
)


In [ ]:
valor_acao.show(truncate=False)

In [ ]:
lucro_acao = (
    lucro_acao
        .withColumn(
                "LPA - total", 
                when(col("ESCALA_MOEDA") == 'MIL', col("LPA - total")*1000)
                .otherwise(col("LPA - total"))
        )
        .withColumn(
                "LPA - pre", 
                when(col("ESCALA_MOEDA") == 'MIL', col("LPA - pre")*1000)
                .otherwise(col("LPA - pre"))
        )
        .withColumn(
                "LPA - on",
                when(col("ESCALA_MOEDA") == 'MIL', col("LPA - on")*1000)
                .otherwise(col("LPA - on"))
        )
)

valor_acao = (
    valor_acao
        .withColumn(
                "VPA - total", 
                when(col("ESCALA_MOEDA") == 'MIL', col("VPA - total")*1000)
                .otherwise(col("VPA - total"))
        )
        .withColumn(
                "VPA - pre", 
                when(col("ESCALA_MOEDA") == 'MIL', col("VPA - pre")*1000)
                .otherwise(col("VPA - pre"))
        )
        .withColumn(
                "VPA - on",
                when(col("ESCALA_MOEDA") == 'MIL', col("VPA - on")*1000)
                .otherwise(col("VPA - on"))
        )
)

In [ ]:
valor_acao.show(truncate=False)

In [ ]:
lucro_acao.show(truncate=False)